In [79]:
import os
import re
import numpy as np
from collections import Counter

In [80]:
def preprocess_text(text):
    """
    Preprocess text by normalizing and tokenizing it into words.

    This function converts text to lowercase, removes punctuation (preserving
    apostrophes within words), normalizes whitespace, and splits the text into
    individual words.

    Parameters
    ----------
    text : str
        The input text string to be preprocessed.

    Returns
    -------
    list
        A list of words (tokens) extracted from the preprocessed text.

    Notes
    -----
    The preprocessing steps include:
    - Converting to lowercase
    - Removing all punctuation except apostrophes
    - Normalizing whitespace (multiple spaces become single spaces)
    - Splitting the text into individual words
    """
    text = text.lower()
    # Keep apostrophes within words, remove other punctuation
    text = re.sub(r'[^\w\s\']', '', text)
    # Replace multiple spaces with single space
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

In [81]:
def load_data(data_files, encoding='latin-1'):
    """
    Load and preprocess tercets from Divine Comedy text files.

    This function reads the text files containing tercets from the three canticas of
    Dante's Divine Comedy (Inferno, Purgatorio, and Paradiso), preprocesses each line
    using the preprocess_text function, and returns lists of word tokens for each cantica.

    Parameters
    ----------
    data_files : dict
        Dictionary mapping cantica names to file paths.
        Expected keys are 'Inferno', 'Purgatorio', and 'Paradiso'.
    encoding : str, optional
        Character encoding to use when reading text files (default: 'latin-1').
        Latin-1 encoding is used instead of UTF-8 to handle special characters in the Italian text.

    Returns
    -------
    tuple
        A tuple containing three lists (inferno_tercets, purgatorio_tercets, paradiso_tercets),
        where each list contains preprocessed tercets as lists of words.

    Notes
    -----
    Each tercet is represented as a list of words after preprocessing, which includes
    lowercasing, punctuation removal, and tokenization.
    """
    data = {}
    for name, path in data_files.items():
        try:
            with open(path, 'r', encoding=encoding) as f:
                # Read lines, preprocess, and filter out empty results
                data[name] = [preprocess_text(line) for line in f if line.strip()]
                # Filter out tercets that became empty after preprocessing
                data[name] = [tercet for tercet in data[name] if tercet]
        except FileNotFoundError:
            print(f"Error: File not found at {path}")
            data[name] = []
        except Exception as e:
            print(f"An error occurred while reading {path}: {e}")
            data[name] = []
    return data['Inferno'], data['Purgatorio'], data['Paradiso']

In [82]:
def split_data(inferno_tercets, purgatorio_tercets, paradiso_tercets, train_ratio=0.75, seed=42):
    """
    Split tercets from each cantica of Divine Comedy into training and validation sets.

    This function takes lists of preprocessed tercets from Inferno, Purgatorio, and
    Paradiso, and randomly splits them into training and validation sets according
    to the specified ratio. It maintains the same random split across executions by
    using a fixed random seed.

    Parameters
    ----------
    inferno_tercets : list
        List of preprocessed tercets from Inferno, each tercet is a list of words.
    purgatorio_tercets : list
        List of preprocessed tercets from Purgatorio, each tercet is a list of words.
    paradiso_tercets : list
        List of preprocessed tercets from Paradiso, each tercet is a list of words.
    train_ratio : float, optional
        The proportion of data to use for training (default: 0.75).
    seed : int, optional
        Random seed for reproducibility (default: 42).

    Returns
    -------
    tuple
        A tuple of two dictionaries (train_data, val_data), each containing lists of
        tercets for each cantica under keys 'Inferno', 'Purgatorio', and 'Paradiso'.
    """
    np.random.seed(seed)  # Set random seed for reproducible splits

    train_data = {'Inferno': [], 'Purgatorio': [], 'Paradiso': []}
    val_data = {'Inferno': [], 'Purgatorio': [], 'Paradiso': []}

    for name, tercets in zip(CLASS_NAMES, [inferno_tercets, purgatorio_tercets, paradiso_tercets]):
        n = len(tercets)
        n_train = int(n * train_ratio)  # Calculate number of training samples
        indices = np.random.permutation(n)  # Create randomly shuffled indices
        train_indices = indices[:n_train]  # Select first portion for training
        val_indices = indices[n_train:]  # Select remaining portion for validation

        # Extract tercets based on the selected indices
        train_data[name] = [tercets[i] for i in train_indices]
        val_data[name] = [tercets[i] for i in val_indices]

    return train_data, val_data

In [83]:
# List of canticas (sections) of Dante's Divine Comedy
CLASS_NAMES = ["Inferno", "Purgatorio", "Paradiso"]

# File paths for each cantica's text content
DATA_FILES = {
    "Inferno": "data/inferno.txt",
    "Purgatorio": "data/purgatorio.txt",
    "Paradiso": "data/paradiso.txt"
}

# Load and preprocess all tercets from each text file
# The load_data function returns lists of preprocessed tercets for each cantica
inferno, purgatorio, paradiso = load_data(DATA_FILES)
print(f"Loaded {len(inferno)} Inferno, {len(purgatorio)} Purgatorio, {len(paradiso)} Paradiso tercets.")

# Split data into training (75%) and validation (25%) sets
# Data is shuffled with a fixed random seed for reproducibility
train_set, val_set = split_data(inferno, purgatorio, paradiso)

# Display the size of each dataset to verify the split was performed correctly
print(f"Training set sizes: Inferno={len(train_set['Inferno'])}, Purgatorio={len(train_set['Purgatorio'])}, Paradiso={len(train_set['Paradiso'])}")
print(f"Validation set sizes: Inferno={len(val_set['Inferno'])}, Purgatorio={len(val_set['Purgatorio'])}, Paradiso={len(val_set['Paradiso'])}")

Loaded 1597 Inferno, 1608 Purgatorio, 1607 Paradiso tercets.
Training set sizes: Inferno=1197, Purgatorio=1206, Paradiso=1205
Validation set sizes: Inferno=400, Purgatorio=402, Paradiso=402


In [84]:
def build_dictionary_and_mapping(train_data):
    """
    Build a dictionary of unique words from all training tercets and mapping dictionaries.

    Creates a vocabulary from all words in the training data and establishes
    bidirectional mappings between words and their indices.

    Parameters
    ----------
    train_data : dict
        Dictionary containing training data organized by class names.
        Expected keys are 'Inferno', 'Purgatorio', and 'Paradiso',
        with each value being a list of tercets (lists of words).

    Returns
    -------
    tuple
        A tuple containing three elements:
        - dictionary (list): Sorted list of unique words from the training data
        - word_to_idx (dict): Dictionary mapping words to their indices
        - idx_to_word (dict): Dictionary mapping indices to their words
    """
    all_words = []
    for class_name in CLASS_NAMES:
        for tercet in train_data[class_name]:
            all_words.extend(tercet)
    # Create a sorted list of unique words for consistent mapping
    dictionary = sorted(list(set(all_words)))
    word_to_idx = {word: idx for idx, word in enumerate(dictionary)}
    idx_to_word = {idx: word for word, idx in word_to_idx.items()}
    return dictionary, word_to_idx, idx_to_word

In [85]:
def calculate_word_counts_and_totals(train_data, word_to_idx):
    """
    Calculate word occurrence counts and total word counts for each class.

    For each class (cantica), this function counts how many times each word
    appears and the total number of words, which are required for Naive Bayes
    probability calculations.

    Parameters
    ----------
    train_data : dict
        Dictionary containing training data organized by class names.
        Each class name maps to a list of tercets, where each tercet is a list of words.
    word_to_idx : dict
        Dictionary mapping words to their corresponding indices in the vocabulary.

    Returns
    -------
    tuple
        A tuple containing two dictionaries:
        - class_word_counts (dict): For each class, contains a numpy array of word counts
          where the index corresponds to the word's index in the dictionary
        - class_total_words (dict): For each class, contains the total number of words
    """
    M = len(word_to_idx)
    class_word_counts = {name: np.zeros(M, dtype=int) for name in CLASS_NAMES}
    class_total_words = {name: 0 for name in CLASS_NAMES}

    for class_name in CLASS_NAMES:
        word_counter = Counter()
        total_words = 0
        for tercet in train_data[class_name]:
            word_counter.update(tercet)
            total_words += len(tercet)

        class_total_words[class_name] = total_words
        for word, count in word_counter.items():
            if word in word_to_idx: # Only count words in the dictionary
                idx = word_to_idx[word]
                class_word_counts[class_name][idx] += count

    return class_word_counts, class_total_words

In [86]:
def estimate_log_probabilities(class_word_counts, class_total_words, dictionary_size, epsilon=1.0):
    """
    Estimate smoothed log probabilities for each word in each class.

    Applies add-epsilon (or Laplace) smoothing and calculates the logarithm of
    probabilities P(word|class) for each word and class combination.
    Using log probabilities helps prevent numerical underflow in Naive Bayes.

    Parameters
    ----------
    class_word_counts : dict
        Dictionary mapping class names to arrays of word counts.
        Each array has length equal to the vocabulary size.
    class_total_words : dict
        Dictionary mapping class names to total word counts for that class.
    dictionary_size : int
        Size of the vocabulary (number of unique words).
    epsilon : float, optional
        Smoothing parameter (default: 1.0 for Laplace smoothing).
        Controls the amount of probability mass assigned to unseen words.

    Returns
    -------
    dict
        Dictionary mapping class names to arrays of log probabilities.
        Each array has length equal to vocabulary size, where each element is
        the log probability of the corresponding word given the class.
    """
    log_probs = {}
    M = dictionary_size
    for class_name in CLASS_NAMES:
        counts = class_word_counts[class_name] # Nc,j array
        total_words = class_total_words[class_name] # Nc

        # Apply smoothing
        smoothed_counts = counts + epsilon
        smoothed_total = total_words + M * epsilon

        # Calculate log probabilities
        log_probs[class_name] = np.log(smoothed_counts / smoothed_total)

    return log_probs

In [87]:
# Builds the vocabulary and word mappings between words and indices
dictionary, word_to_idx, idx_to_word = build_dictionary_and_mapping(train_set)
M = len(dictionary)
print(f"Built dictionary with {M} unique words.")

# Calculates word frequencies for each class
class_word_counts, class_total_words = calculate_word_counts_and_totals(train_set, word_to_idx)
print("Calculated word counts per class:")
for name in CLASS_NAMES:
    print(f"  {name}: {class_total_words[name]} total words")

# Sets the smoothing parameter and calculates log probabilities
epsilon = 0.001 # As suggested in the text for expected results
epsilon = 1.0 # Laplace smoothing
log_probabilities = estimate_log_probabilities(class_word_counts, class_total_words, M, epsilon=epsilon)
print(f"Calculated log probabilities with epsilon={epsilon}")

Built dictionary with 11374 unique words.
Calculated word counts per class:
  Inferno: 24551 total words
  Purgatorio: 24441 total words
  Paradiso: 23937 total words
Calculated log probabilities with epsilon=1.0
